# Named Entity Recognition (NER) Fine-Tuning pada Bahasa Indonesia

## Deskripsi Proyek

Notebook ini mengimplementasikan **Named Entity Recognition (NER)** — tugas NLP untuk mengidentifikasi dan mengklasifikasikan entitas bernama dalam teks (seperti nama orang, organisasi, dan lokasi) — menggunakan teknik **fine-tuning** pada model bahasa besar berbasis Transformer.

---

## Konsep Utama

### Apa itu NER?
**Named Entity Recognition (NER)** adalah tugas klasifikasi token-level dalam Natural Language Processing (NLP) yang bertujuan mendeteksi dan mengkategorikan entitas bernama dalam teks.

**Contoh:**
> *"Sri Mulyani Indrawati bekerja di Kementerian Keuangan, Jakarta."*
> - `Sri Mulyani Indrawati` → **PER** (Person)
> - `Kementerian Keuangan` → **ORG** (Organization)
> - `Jakarta` → **LOC** (Location)

### Skema Label IOB2
Setiap token diberi label menggunakan format **IOB2 (Inside-Outside-Beginning)**:
| Label | Arti | Contoh |
|-------|------|--------|
| `O` | Outside — bukan entitas | kata biasa |
| `B-XXX` | Beginning — awal entitas tipe XXX | `B-PER`, `B-LOC`, `B-ORG` |
| `I-XXX` | Inside — lanjutan entitas tipe XXX | `I-PER`, `I-LOC`, `I-ORG` |

### Fine-Tuning
**Fine-tuning** adalah proses melatih ulang model pretrained pada dataset spesifik dengan sedikit epoch dan learning rate kecil, sehingga model belajar tugas baru tanpa kehilangan pengetahuan umum yang sudah dimiliki.

---

## Struktur Eksperimen

Notebook ini berisi **3 eksperimen** yang membandingkan kombinasi model dan dataset:

| Eksperimen | Model | Dataset |
|------------|-------|---------|
| 1 | IndoBERT (`indobert-base-p1`) | WikiANN (ID) |
| 2 | IndoBERT (`indobert-base-p1`) | NER-Grit (IndoNLU) |
| 3 | XLM-RoBERTa-Large (`cahya`) | NER-Grit (IndoNLU) |


---

# Bagian 1 — NER Fine-Tuning dengan Dataset WikiANN (IndoBERT)

## Gambaran Umum

**IndoBERT** adalah model BERT yang dilatih khusus pada korpus teks berbahasa Indonesia oleh IndoBenchmark. Model ini menggunakan arsitektur BERT-base (12 layer, 768 hidden units, ~124M parameter) dengan tokenizer **WordPiece**.

**WikiANN** adalah dataset NER multibahasa yang dibangun dari Wikipedia, tersedia untuk bahasa Indonesia (`id`). Dataset ini mencakup entitas:
- `PER` — Person (Nama orang)
- `ORG` — Organization (Nama organisasi)
- `LOC` — Location (Nama lokasi)

---

## Langkah 1.1 — Instalasi Library

Library yang digunakan:
- `transformers` — framework utama untuk model Transformer (Hugging Face)
- `datasets` — pengambilan dan pengelolaan dataset
- `seqeval` — evaluasi metrik NER (precision, recall, F1)
- `accelerate` — akselerasi pelatihan
- `evaluate` — antarmuka metrik evaluasi


In [1]:
!pip install -q transformers[torch] datasets seqeval accelerate evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


## Langkah 1.2 — Import Library

Modul utama yang diimpor:
- `numpy` — komputasi numerik
- `evaluate` — memuat metrik seqeval
- `torch` — framework deep learning PyTorch
- `AutoTokenizer` — tokenizer otomatis sesuai model
- `AutoModelForTokenClassification` — model klasifikasi token
- `DataCollatorForTokenClassification` — penggabungan batch dengan padding dinamis
- `TrainingArguments` & `Trainer` — konfigurasi dan eksekusi pelatihan


In [2]:
import numpy as np
import evaluate
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

## Langkah 1.3 — Load Dataset dan Konfigurasi Label

Dataset WikiANN dimuat dari Hugging Face Hub untuk bahasa Indonesia.

**Label list WikiANN:**
```
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
```

Dua mapping dibuat:
- `id2label` — integer → nama label (untuk decoding prediksi)
- `label2id` — nama label → integer (untuk encoding input)


In [3]:
# ── Load dataset WikiANN bahasa Indonesia ──────────────────────────────────
MODEL_CHECKPOINT = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

dataset = load_dataset("wikiann", "id")

# Ambil label langsung dari fitur dataset (urutan sudah benar)
label_list = dataset["train"].features["ner_tags"].feature.names
# label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"Labels: {label_list}")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

id/train-00000-of-00001.parquet:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


## Langkah 1.4 — Tokenisasi dan Alignment Label

### Masalah Sub-token
Tokenizer BERT memecah kata menjadi **sub-token** (WordPiece). Satu kata bisa menjadi beberapa token. Karena label NER diberikan per-kata, kita perlu **menyelaraskan (align)** label ke sub-token.

**Strategi alignment:**
- Token pertama dari setiap kata → **label asli**
- Sub-token berikutnya dari kata yang sama:
  - Jika label asli `B-XXX` → diubah ke `I-XXX` (konsistensi IOB2)
  - Jika label asli `I-XXX` atau `O` → tetap sama
- Token spesial (`[CLS]`, `[SEP]`, padding) → `-100` (diabaikan oleh loss function)


In [4]:
def tokenize_and_align_labels(examples):
    """
    Tokenisasi + alignment label ke sub-token.
    - Token pertama dari setiap kata → label asli
    - Sub-token selanjutnya dari kata yang sama:
        jika label asli B-XXX → di-convert ke I-XXX (IOB2 konsisten)
        jika label asli I-XXX / O → tetap sama
    - Token special ([CLS], [SEP], padding) → -100 (diabaikan loss)
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )

    all_labels = []
    for i, label_ids_orig in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Token special → abaikan
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Token pertama dari kata → pakai label asli
                label_ids.append(label_ids_orig[word_idx])
            else:
                # Sub-token (bukan token pertama dari kata)
                orig_label_id = label_ids_orig[word_idx]
                orig_label_name = label_list[orig_label_id]
                # Jika label aslinya B-XXX, sub-token harus I-XXX
                if orig_label_name.startswith("B-"):
                    i_label = "I-" + orig_label_name[2:]
                    label_ids.append(label2id[i_label])
                else:
                    label_ids.append(orig_label_id)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print(tokenized_datasets)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


## Langkah 1.5 — Fungsi Evaluasi (compute_metrics)

Evaluasi menggunakan library **seqeval** dengan mode `strict` dan skema `IOB2`.

**Metrik yang dihitung:**
| Metrik | Definisi |
|--------|----------|
| **Precision** | Dari semua prediksi entitas, berapa persen yang benar |
| **Recall** | Dari semua entitas ground truth, berapa persen yang terdeteksi |
| **F1-Score** | Rata-rata harmonik precision dan recall |
| **Accuracy** | Akurasi per-token |

> F1-Score adalah metrik utama untuk NER karena menyeimbangkan precision dan recall.


In [5]:
seqeval = evaluate.load("seqeval")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lbl] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
        mode="strict",
        scheme="IOB2"
    )
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

## Langkah 1.6 — Inisialisasi Model dan Pelatihan

### Model
`AutoModelForTokenClassification` dimuat dari checkpoint IndoBERT dan dikonfigurasi dengan jumlah label NER yang sesuai.

### Training Arguments (Hyperparameter)
| Parameter | Nilai | Penjelasan |
|-----------|-------|------------|
| `learning_rate` | 2e-5 | Learning rate kecil untuk fine-tuning stabil |
| `num_train_epochs` | 5 | Jumlah epoch pelatihan |
| `per_device_train_batch_size` | 16 | Ukuran batch per GPU/CPU |
| `weight_decay` | 0.01 | Regularisasi L2 untuk mencegah overfitting |
| `metric_for_best_model` | f1 | Model terbaik dipilih berdasarkan F1 |
| `fp16` | auto | Mixed precision jika GPU tersedia |

### Data Collator
`DataCollatorForTokenClassification` melakukan padding dinamis — setiap batch dipadding hingga panjang terpanjang dalam batch, bukan panjang maksimum global, sehingga lebih efisien.


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./indobert-wikiann-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.395979,0.389633,0.674803,0.635466,0.654544,0.862669
2,0.342554,0.375333,0.695407,0.690788,0.693090,0.869542
3,0.299142,0.360031,0.713958,0.706345,0.710131,0.877841
4,0.256339,0.370649,0.690279,0.686418,0.688344,0.879954
5,0.228188,0.374038,0.693985,0.689740,0.691856,0.880832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6250, training_loss=0.320942336730957, metrics={'train_runtime': 606.5522, 'train_samples_per_second': 164.866, 'train_steps_per_second': 10.304, 'total_flos': 967976762825952.0, 'train_loss': 0.320942336730957, 'epoch': 5.0})

## Langkah 1.7 — Inferensi (Single Text)

Model diuji menggunakan `pipeline` Hugging Face dengan `aggregation_strategy="simple"` yang menggabungkan sub-token menjadi entitas utuh.

**Teks uji:** Artikel tentang pajak hiburan, Hotman Paris Hutapea, dan Sri Mulyani Indrawati.


In [11]:
from transformers import pipeline

ner_pipeline_wikiann = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sample_text = (
    "Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada "
    "Menteri Keuangan, Sri Mulyani Indrawati. Hal ini terkait kenaikan pajak "
    "hiburan yang memberatkan para pengusaha hiburan Indonesia. "
    "Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta."
)

results = ner_pipeline_wikiann(sample_text)
for entity in results:
    print(f"Entity: {entity['word']:<30} | Label: {entity['entity_group']:<6} | Score: {entity['score']:.4f}")

Entity: Beach Club Atlas               | Label: PER    | Score: 0.6918
Entity: Hotman Paris Hutapea           | Label: PER    | Score: 0.6962
Entity: Menteri Keuangan               | Label: PER    | Score: 0.9000
Entity: Sri Mulyani Indrawati.         | Label: PER    | Score: 0.5820
Entity: Indonesia. Kantor Kementerian Koordinator Bidang Kemaritiman | Label: ORG    | Score: 0.6610
Entity: Investasi                      | Label: ORG    | Score: 0.3198


## Langkah 1.8 — Inferensi (Multi-Kalimat)

Artikel dibagi per kalimat untuk menghindari batas 512 token BERT dan menjaga konteks bersih sesuai cara model dilatih.

Hasil ditampilkan per kalimat dengan format: `label | entitas | confidence score`.


In [13]:
from transformers import pipeline

ner_pipeline_wikiann = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# Artikel dipecah per kalimat agar tidak melebihi batas 512 token
# dan agar konteks tiap kalimat tetap bersih sesuai cara model dilatih
sample_sentences = [
    "Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.",
    "Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.",
    "Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung Kementerian Keuangan, Jakarta Pusat.",
    "Aturan kenaikan pajak hiburan tercantum dalam Undang-Undang Nomor 1 Tahun 2022 tentang Hubungan Keuangan antara Pemerintah Pusat dan Pemerintahan Daerah.",
    "Direktorat Jenderal Perimbangan Keuangan (DJPK) Kementerian Keuangan menyebut pengenaan besaran pajak tersebut karena penikmat hiburan karaoke hingga spa berasal dari masyarakat kalangan tertentu.",
    "Pembahasaan UU HKPD terkait pengenaan besaran pajak itu telah disetujui dan disepakati oleh Dewan Perwakilan Rakyat (DPR) RI.",
]

print("=" * 70)
print(f"{'Model: IndoBERT + WikiANN':^70}")
print("=" * 70)

for i, sent in enumerate(sample_sentences, 1):
    results = ner_pipeline_wikiann(sent)
    print(f"\n[Kalimat {i}] {sent}")
    if results:
        for ent in results:
            print(f"  → {ent['entity_group']:<6}  {ent['word']:<35}  score: {ent['score']:.4f}")
    else:
        print("  → (tidak ada entitas terdeteksi)")

                      Model: IndoBERT + WikiANN                       

[Kalimat 1] Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.
  → PER     Beach Club Atlas                     score: 0.8512
  → PER     Hotman Paris Hutapea                 score: 0.9357
  → PER     Menteri Keuangan                     score: 0.9855
  → PER     Sri Mulyani Indrawati                score: 0.9534

[Kalimat 2] Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.
  → PER     Hotman                               score: 0.5931
  → ORG     Kantor Kementerian Koordinator Bidang Kemaritiman  score: 0.7156
  → LOC     Investasi                            score: 0.6168
  → LOC     Jakarta                              score: 0.6786

[Kalimat 3] Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung Kementerian Keuangan, Jakarta Pusat.
  → PER     

## Langkah 1.9 — Simpan Model

Model terbaik (berdasarkan F1 validasi) disimpan ke direktori `./indobert-wikiann-best/` untuk digunakan kembali tanpa perlu melatih ulang.


In [ ]:
trainer.save_model("./indobert-wikiann-best")
tokenizer.save_pretrained("./indobert-wikiann-best")
print("Model WikiANN tersimpan.")

---

# Bagian 2 — NER Fine-Tuning: NER-Grit (IndoNLU) + IndoBERT

## Gambaran Umum

Eksperimen kedua menggunakan dataset **NER-Grit** dari repositori **IndoNLU** — benchmark NLP Indonesia yang lebih domain-spesifik dibanding WikiANN.

### Perbedaan NER-Grit vs WikiANN
| Aspek | WikiANN | NER-Grit |
|-------|---------|----------|
| Sumber | Wikipedia | Berita Indonesia |
| Format label | Standar PER/LOC/ORG | PERSON, PLACE, ORGANISATION |
| Format file | Hugging Face Hub | File `.txt` lokal |
| Jumlah label | 7 | Lebih banyak (perlu normalisasi) |

Dataset NER-Grit menggunakan nama label yang berbeda sehingga perlu **mapping eksplisit** ke format standar.

---

## Langkah 2.1 — Instalasi dan Import


In [2]:
!pip install -q transformers[torch] datasets seqeval accelerate evaluate

In [3]:
import numpy as np
import evaluate
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

## Langkah 2.2 — Clone Repository IndoNLU

Repository IndoNLU di-clone dari GitHub untuk mendapatkan akses ke dataset NER-Grit dalam format file teks.


In [ ]:
!git clone https://github.com/indobenchmark/indonlu.git

In [ ]:
import os

base_path = "/content/indonlu/dataset"

print(os.listdir(base_path))

In [27]:
ner_path2 = f"{base_path}/nergrit_ner-grit"
print(os.listdir(ner_path2))

['vocab_uncased.txt', 'valid_preprocess.txt', 'test_preprocess.txt', 'test_preprocess_masked_label.txt', 'vocab.txt', 'train_preprocess.txt']


In [34]:
import os

ner_path = "/content/indonlu/dataset/nergrit_ner-grit"

print(os.listdir(ner_path))

['vocab_uncased.txt', 'valid_preprocess.txt', 'test_preprocess.txt', 'test_preprocess_masked_label.txt', 'vocab.txt', 'train_preprocess.txt']


## Langkah 2.3 — Eksplorasi Format File Dataset

File dataset NER-Grit berformat **CoNLL** — setiap baris berisi satu token dan labelnya, dipisahkan oleh baris kosong antar kalimat.

**Contoh format:**
```
Hotman   B-PERSON
Paris    I-PERSON
Jakarta  B-PLACE
adalah   O
```


In [36]:
train_file = f"{ner_path}/train_preprocess.txt"

with open(train_file, "r", encoding="utf-8") as f:
    for i in range(30):
        print(repr(f.readline()))

'Kontribusinya\tO\n'
'terhadap\tO\n'
'industri\tO\n'
'musik\tO\n'
'telah\tO\n'
'mengumpulkan\tO\n'
'banyak\tO\n'
'prestasi\tO\n'
'termasuk\tO\n'
'lima\tO\n'
'Grammy\tO\n'
'Awards\tO\n'
',\tO\n'
'serta\tO\n'
'dua\tO\n'
'belas\tO\n'
'nominasi\tO\n'
';\tO\n'
'dua\tO\n'
'Guinness\tO\n'
'World\tO\n'
'Records\tO\n'
';\tO\n'
'dan\tO\n'
'penjualannya\tO\n'
'diperkirakan\tO\n'
'sekitar\tO\n'
'64\tO\n'
'juta\tO\n'
'rekaman\tO\n'


## Langkah 2.4 — Fungsi Load Dataset (Tab-delimited)

Fungsi `load_ner_dataset()` membaca file CoNLL baris per baris:
- Baris berisi token → dipecah berdasarkan tab, ambil token pertama dan label terakhir
- Baris kosong → penanda batas antar kalimat
- Output: daftar kalimat (list of tokens) dan daftar label (list of tags)


In [48]:
def load_ner_dataset(path):

    sentences = []
    labels = []

    tokens = []
    tags = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            # separator kalimat
            if line == "":

                if len(tokens) > 0:
                    sentences.append(tokens)
                    labels.append(tags)

                tokens = []
                tags = []

            else:
                splits = line.split()

                token = splits[0]
                tag = splits[-1]

                tokens.append(token)
                tags.append(tag)

    return sentences, labels

In [39]:
train_sentences, train_labels = load_ner_dataset(
    f"{ner_path}/train_preprocess.txt"
)

valid_sentences, valid_labels = load_ner_dataset(
    f"{ner_path}/valid_preprocess.txt"
)

test_sentences, test_labels = load_ner_dataset(
    f"{ner_path}/test_preprocess.txt"
)

In [19]:
ner_path = "/content/indonlu/dataset/nerp_ner-prosa"
print(os.listdir(ner_path))

['vocab_uncased.txt', 'valid_preprocess.txt', 'test_preprocess.txt', 'test_preprocess_masked_label.txt', 'vocab.txt', 'train_preprocess.txt']


In [40]:
print("Jumlah train:", len(train_sentences))

print("\nSample:\n")

for token, label in zip(train_sentences[0], train_labels[0]):
    print(f"{token:15} -> {label}")

Jumlah train: 1672

Sample:

Kontribusinya   -> O
terhadap        -> O
industri        -> O
musik           -> O
telah           -> O
mengumpulkan    -> O
banyak          -> O
prestasi        -> O
termasuk        -> O
lima            -> O
Grammy          -> O
Awards          -> O
,               -> O
serta           -> O
dua             -> O
belas           -> O
nominasi        -> O
;               -> O
dua             -> O
Guinness        -> O
World           -> O
Records         -> O
;               -> O
dan             -> O
penjualannya    -> O
diperkirakan    -> O
sekitar         -> O
64              -> O
juta            -> O
rekaman         -> O
.               -> O


In [41]:
unique_labels = sorted(
    set(label for sent in train_labels for label in sent)
)

print(unique_labels)

['B-ORGANISATION', 'B-PERSON', 'B-PLACE', 'I-ORGANISATION', 'I-PERSON', 'I-PLACE', 'O']


## Langkah 2.5 — Load Dataset NER-Grit (Tab-split) dan Eksplorasi Label

Dataset NER-Grit aslinya menggunakan tab sebagai pemisah (berbeda dengan spasi pada eksplorasi sebelumnya). Fungsi `load_ner_dataset` disesuaikan menggunakan `line.split("\t")`.

**Label unik NER-Grit yang ditemukan:**
- `B-PERSON`, `I-PERSON` → akan dimap ke `B-PER`, `I-PER`
- `B-ORGANISATION`, `I-ORGANISATION` → akan dimap ke `B-ORG`, `I-ORG`
- `B-PLACE`, `I-PLACE` → akan dimap ke `B-LOC`, `I-LOC`
- Label lain → dimap ke `O`


In [30]:
train_file = f"{ner_path2}/train_preprocess.txt"

with open(train_file, "r", encoding="utf-8") as f:
    for i in range(30):
        print(repr(f.readline()))

'Kontribusinya\tO\n'
'terhadap\tO\n'
'industri\tO\n'
'musik\tO\n'
'telah\tO\n'
'mengumpulkan\tO\n'
'banyak\tO\n'
'prestasi\tO\n'
'termasuk\tO\n'
'lima\tO\n'
'Grammy\tO\n'
'Awards\tO\n'
',\tO\n'
'serta\tO\n'
'dua\tO\n'
'belas\tO\n'
'nominasi\tO\n'
';\tO\n'
'dua\tO\n'
'Guinness\tO\n'
'World\tO\n'
'Records\tO\n'
';\tO\n'
'dan\tO\n'
'penjualannya\tO\n'
'diperkirakan\tO\n'
'sekitar\tO\n'
'64\tO\n'
'juta\tO\n'
'rekaman\tO\n'


In [31]:
def load_ner_dataset(path):

    sentences = []
    labels = []

    tokens = []
    tags = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            # pemisah kalimat
            if line == "":

                if len(tokens) > 0:
                    sentences.append(tokens)
                    labels.append(tags)

                tokens = []
                tags = []

            else:
                splits = line.split("\t")

                # token dan label
                token = splits[0]
                tag = splits[-1]

                tokens.append(token)
                tags.append(tag)

    return sentences, labels

In [32]:
train_sentences, train_labels = load_ner_dataset(
    f"{ner_path}/train_preprocess.txt"
)

valid_sentences, valid_labels = load_ner_dataset(
    f"{ner_path}/valid_preprocess.txt"
)

test_sentences, test_labels = load_ner_dataset(
    f"{ner_path}/test_preprocess.txt"
)

In [42]:
# ── Cek label unik NER-Grit ────────────────────────────────────────────────
unique_labels_grit = sorted(
    set(label for sent in train_labels for label in sent)
)
print("Label unik NER-Grit:", unique_labels_grit)


Label unik NER-Grit: ['B-ORGANISATION', 'B-PERSON', 'B-PLACE', 'I-ORGANISATION', 'I-PERSON', 'I-PLACE', 'O']


## Langkah 2.6 — Bangun HuggingFace DatasetDict dan Label Mapping

### Label Normalisasi
Karena NER-Grit menggunakan nama label yang berbeda dari format standar, dibuat **mapping eksplisit**:

```python
GRIT_LABEL_MAP = {
    "B-PERSON":       "B-PER",
    "I-PERSON":       "I-PER",
    "B-ORGANISATION": "B-ORG",
    "I-ORGANISATION": "I-ORG",
    "B-PLACE":        "B-LOC",
    "I-PLACE":        "I-LOC",
    "O":              "O",
}
```

### HuggingFace DatasetDict
Data yang dimuat dari file teks dikonversi menjadi `DatasetDict` format Hugging Face agar bisa digunakan oleh `Trainer` — format yang konsisten dengan WikiANN.


In [50]:
# ── Bangun HuggingFace DatasetDict ────────────────────────────────────────
import numpy as np
import evaluate
import torch

from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

# ── Label list output (standar PER/LOC/ORG) ──────────────────────────────
label_list_grit = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
id2label_grit   = {i: lbl for i, lbl in enumerate(label_list_grit)}
label2id_grit   = {lbl: i for i, lbl in enumerate(label_list_grit)}

# ── Mapping dari label NER-Grit asli ke label standar ────────────────────
# NER-Grit menggunakan nama label: B-PERSON, B-PLACE, B-ORGANIZATION, dsb.
# Kita normalisasi ke format PER/LOC/ORG. Label yang tidak relevan → O.
GRIT_LABEL_MAP = {
    "O":               "O",
    "B-PERSON":        "B-PER",
    "I-PERSON":        "I-PER",
    "B-ORGANISATION":  "B-ORG",
    "I-ORGANISATION":  "I-ORG",
    "B-PLACE":         "B-LOC",
    "I-PLACE":         "I-LOC",
    # label lain yang mungkin ada di NER-Grit → abaikan (jadi O)
}

def map_grit_label(raw_label):
    """Konversi label NER-Grit asli ke label standar, default 'O' jika tidak dikenal."""
    std_label = GRIT_LABEL_MAP.get(raw_label, "O")
    return label2id_grit[std_label]

features_grit = Features({
    "tokens":   Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=label_list_grit)),
})

def build_split(sentences, labels):
    return Dataset.from_dict(
        {
            "tokens":   sentences,
            "ner_tags": [
                [map_grit_label(t) for t in sent_labels]  # FIX: pakai mapping eksplisit
                for sent_labels in labels
            ],
        },
        features=features_grit,
    )

dataset_grit = DatasetDict({
    "train":      build_split(train_sentences, train_labels),
    "validation": build_split(valid_sentences, valid_labels),
    "test":       build_split(test_sentences,  test_labels),
})

print(dataset_grit)
print("\nLabel list:", label_list_grit)

# ── Verifikasi: pastikan ada token entitas (non-O) ────────────────────────
from collections import Counter
all_raw = [t for sent in train_labels for t in sent]
raw_counter = Counter(all_raw)
print("\n── Raw label dari file (sebelum mapping) ──")
for lbl, cnt in sorted(raw_counter.items()):
    mapped = GRIT_LABEL_MAP.get(lbl, "→ O (unmapped)")
    print(f"  {lbl:<20} : {cnt:>7,}  →  {mapped}")


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1672
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 209
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 209
    })
})

Label list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

── Raw label dari file (sebelum mapping) ──
  B-ORGANISATION       :   1,015  →  B-ORG
  B-PERSON             :   1,618  →  B-PER
  B-PLACE              :   2,688  →  B-LOC
  I-ORGANISATION       :   1,204  →  I-ORG
  I-PERSON             :   1,305  →  I-PER
  I-PLACE              :   2,288  →  I-LOC
  O                    :  46,092  →  O


## Langkah 2.7 — Tokenisasi dan Alignment Label (IndoBERT)

Logika tokenisasi sama dengan Bagian 1, namun menggunakan `tokenizer_grit` yang dimuat khusus untuk eksperimen ini dan label list NER-Grit yang sudah dinormalisasi.

Setelah tokenisasi, dilakukan **verifikasi**: memastikan ada token entitas (non-O) dalam dataset agar pelatihan bermakna.


In [51]:
# ── Tokenisasi + alignment label ke sub-token ─────────────────────────────
MODEL_CHECKPOINT_GRIT = "indobenchmark/indobert-base-p1"
tokenizer_grit = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT_GRIT)

def tokenize_and_align_labels_grit(examples):
    """
    - Token pertama dari setiap kata -> label asli
    - Sub-token berikutnya: B-XXX -> I-XXX, lainnya tetap
    - Token special ([CLS], [SEP]) -> -100
    """
    tokenized_inputs = tokenizer_grit(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False,
    )

    all_labels = []
    for i, label_ids_orig in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_ids_orig[word_idx])
            else:
                orig_label_id   = label_ids_orig[word_idx]
                orig_label_name = label_list_grit[orig_label_id]
                if orig_label_name.startswith("B-"):
                    label_ids.append(label2id_grit["I-" + orig_label_name[2:]])
                else:
                    label_ids.append(orig_label_id)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_grit = dataset_grit.map(tokenize_and_align_labels_grit, batched=True)
print(tokenized_grit)


Map:   0%|          | 0/1672 [00:00<?, ? examples/s]

Map:   0%|          | 0/209 [00:00<?, ? examples/s]

Map:   0%|          | 0/209 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1672
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 209
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 209
    })
})


## Langkah 2.8 — Fungsi Evaluasi (seqeval)

Sama seperti Bagian 1 — menggunakan seqeval dengan mode `strict` dan skema `IOB2`, namun menggunakan label list NER-Grit yang sudah dinormalisasi.


In [52]:
# ── Metric: seqeval ───────────────────────────────────────────────────────
seqeval_grit = evaluate.load("seqeval")

def compute_metrics_grit(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_grit[pred] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list_grit[lbl] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval_grit.compute(
        predictions=true_predictions,
        references=true_labels,
        mode="strict",
        scheme="IOB2",
    )
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }


## Langkah 2.9 — Model, Training Arguments, dan Pelatihan

Konfigurasi hampir identik dengan Bagian 1:
- Model: IndoBERT (`indobert-base-p1`)
- Learning rate: `2e-5`
- Epochs: 5, Batch size: 16
- Metric terbaik: F1

Perbedaan dari Bagian 1: dataset yang digunakan adalah NER-Grit (bukan WikiANN), sehingga model belajar dari distribusi data berita Indonesia.


In [53]:
# ── Model + TrainingArguments + Trainer ───────────────────────────────────
model_grit = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT_GRIT,
    num_labels=len(label_list_grit),
    id2label=id2label_grit,
    label2id=label2id_grit,
)

data_collator_grit = DataCollatorForTokenClassification(tokenizer_grit)

training_args_grit = TrainingArguments(
    output_dir="./indobert-nergrit-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_grit = Trainer(
    model=model_grit,
    args=training_args_grit,
    train_dataset=tokenized_grit["train"],
    eval_dataset=tokenized_grit["validation"],
    data_collator=data_collator_grit,
    compute_metrics=compute_metrics_grit,
)

trainer_grit.train()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.304714,0.265154,0.667319,0.515106,0.581415,0.909434
2,0.233665,0.246213,0.681261,0.587613,0.630981,0.918519
3,0.184758,0.246774,0.668740,0.649547,0.659004,0.921733
4,0.143916,0.248795,0.694175,0.648036,0.670312,0.924389
5,0.128137,0.260044,0.685535,0.658610,0.671803,0.923270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=525, training_loss=0.22824411800929478, metrics={'train_runtime': 213.8577, 'train_samples_per_second': 39.091, 'train_steps_per_second': 2.455, 'total_flos': 354803957823216.0, 'train_loss': 0.22824411800929478, 'epoch': 5.0})

## Langkah 2.10 — Inferensi dengan Model NER-Grit

Model diuji dengan kalimat-kalimat yang sama seperti Bagian 1 untuk memungkinkan **perbandingan langsung** antara model WikiANN dan NER-Grit.

Output ditampilkan dengan label, entitas yang terdeteksi, dan confidence score.


In [54]:
from transformers import pipeline

ner_pipeline_grit = pipeline(
    "token-classification",
    model=model_grit,
    tokenizer=tokenizer_grit,
    aggregation_strategy="simple"
)

sample_sentences = [
    "Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.",
    "Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.",
    "Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung Kementerian Keuangan, Jakarta Pusat.",
    "Aturan kenaikan pajak hiburan tercantum dalam Undang-Undang Nomor 1 Tahun 2022 tentang Hubungan Keuangan antara Pemerintah Pusat dan Pemerintahan Daerah.",
    "Direktorat Jenderal Perimbangan Keuangan (DJPK) Kementerian Keuangan menyebut pengenaan besaran pajak tersebut karena penikmat hiburan karaoke hingga spa berasal dari masyarakat kalangan tertentu.",
    "Pembahasaan UU HKPD terkait pengenaan besaran pajak itu telah disetujui dan disepakati oleh Dewan Perwakilan Rakyat (DPR) RI.",
]

print("=" * 70)
print(f"{'Model: IndoBERT + NER-Grit':^70}")
print("=" * 70)

for i, sent in enumerate(sample_sentences, 1):
    results = ner_pipeline_grit(sent)
    print(f"\n[Kalimat {i}] {sent}")
    if results:
        for ent in results:
            print(f"  → {ent['entity_group']:<6}  {ent['word']:<35}  score: {ent['score']:.4f}")
    else:
        print("  → (tidak ada entitas terdeteksi)")

                      Model: IndoBERT + NER-Grit                      

[Kalimat 1] Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.
  → PER     Beach Club Atlas                     score: 0.6405
  → PER     Hotman Paris Hutapea                 score: 0.8961
  → PER     Menteri Keuangan                     score: 0.9476
  → PER     Sri Mulyani Indrawati                score: 0.9218

[Kalimat 2] Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.
  → PER     Hotman                               score: 0.4319
  → LOC     Kantor                               score: 0.4661
  → LOC     Kemaritiman                          score: 0.5304
  → LOC     Investasi                            score: 0.6556
  → LOC     Jakarta                              score: 0.8457

[Kalimat 3] Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung 

---

# Bagian 3 — NER Fine-Tuning: NER-Grit + XLM-RoBERTa-Large (cahya)

## Gambaran Umum

Eksperimen ketiga menggunakan model **XLM-RoBERTa-Large** yang telah di-pretrain ulang pada korpus teks Indonesia oleh [cahya](https://huggingface.co/cahya). Ini adalah model paling besar dan paling kuat dalam tiga eksperimen ini.

## Mengapa XLM-RoBERTa-Large (cahya)?

Model `cahya/xlm-roberta-large-indonesian-NER` adalah XLM-RoBERTa-Large yang telah di-pretrain ulang pada korpus teks Indonesia berskala besar.

### Perbandingan Model

| Aspek | IndoBERT (base) | XLM-RoBERTa-Large (cahya) |
|---|---|---|
| Arsitektur | BERT-base (12L, 768H) | RoBERTa-large (24L, 1024H) |
| Parameter | ~124 juta | ~560 juta |
| Tokenizer | WordPiece | SentencePiece (BPE) |
| Bahasa pretrain | Hanya Indonesia | 100 bahasa + korpus Indonesia besar |

Dataset yang digunakan tetap **NER-Grit** (sama dengan Bagian 2) sehingga perbandingan antar model dilakukan secara **apple-to-apple** — hanya model yang berbeda, dataset sama.

---

## Langkah 3.1 — Instalasi Library Tambahan

XLM-RoBERTa memerlukan `sentencepiece` dan `protobuf` karena menggunakan tokenizer SentencePiece (BPE), berbeda dengan WordPiece pada BERT.


In [55]:
!pip install -q transformers[torch] datasets seqeval accelerate evaluate sentencepiece protobuf

## Langkah 3.2 — Import Library dan Cek GPU


In [56]:
import numpy as np
import evaluate
import torch

from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

print("Import selesai.")
print(f"GPU tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Import selesai.
GPU tersedia: True
GPU: Tesla T4


## Langkah 3.3 — Gunakan Ulang Dataset NER-Grit

Dataset `dataset_grit`, label list, dan mapping yang sudah dibuat di Bagian 2 digunakan kembali langsung — tidak perlu memuat ulang. Ini memastikan konsistensi data antar eksperimen.


In [57]:
# ── Gunakan ulang dataset_grit dari Section 2 ────────────────────────────
# Pastikan cell-cell Section 2 sudah dijalankan sebelum cell ini.

# Label list dan mapping sudah ada:
#   label_list_grit, id2label_grit, label2id_grit, dataset_grit
print("dataset_grit:")
print(dataset_grit)
print("\nLabel list:", label_list_grit)

dataset_grit:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1672
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 209
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 209
    })
})

Label list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


## Langkah 3.4 — Load Tokenizer XLM-RoBERTa-Large

**Perbedaan penting dari IndoBERT:**
- XLM-RoBERTa menggunakan **SentencePiece (BPE)** bukan WordPiece
- Sub-token ditandai dengan `▁` (underline) sebagai penanda spasi
- Contoh: `"Hotman"` → `["▁Hot", "man"]`

Ukuran vocabulary lebih besar dibanding IndoBERT karena mencakup 100 bahasa.


In [62]:
# ── Load tokenizer XLM-RoBERTa-Large (cahya) ─────────────────────────────
MODEL_CHECKPOINT_CAHYA = "cahya/xlm-roberta-large-indonesian-NER"
tokenizer_cahya = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT_CAHYA)

print(f"Tokenizer: {tokenizer_cahya.__class__.__name__}")
print(f"Vocab size: {tokenizer_cahya.vocab_size:,}")

# Contoh tokenisasi — XLM-R pakai SentencePiece (▁ sebagai spasi)
sample_tokens = ["Hotman", "Paris", "Hutapea"]
enc = tokenizer_cahya(sample_tokens, is_split_into_words=True)
print(f"\nContoh token → subword: {tokenizer_cahya.convert_ids_to_tokens(enc['input_ids'])}")

Tokenizer: XLMRobertaTokenizer
Vocab size: 250,002

Contoh token → subword: ['<s>', '▁Hot', 'man', '▁Paris', '▁Hu', 'tape', 'a', '</s>']


## Langkah 3.5 — Tokenisasi dan Alignment Label (XLM-RoBERTa)

Logika alignment label **identik** dengan Bagian 1 dan 2:
- Token pertama setiap kata → label asli
- Sub-token berikutnya: `B-XXX` → `I-XXX`
- Token spesial (`<s>`, `</s>`) → `-100`

Setelah tokenisasi, dilakukan verifikasi untuk memastikan ada token entitas.


In [63]:
def tokenize_and_align_labels_cahya(examples):
    """
    Tokenisasi + alignment label untuk XLM-RoBERTa.
    Logika sama dengan IndoBERT:
    - Token pertama setiap kata  → label asli
    - Sub-token berikutnya       → B-XXX diubah ke I-XXX, lainnya tetap
    - Token special (<s>, </s>) → -100
    """
    tokenized_inputs = tokenizer_cahya(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False,
    )

    all_labels = []
    for i, label_ids_orig in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_ids_orig[word_idx])
            else:
                orig_label_id   = label_ids_orig[word_idx]
                orig_label_name = label_list_grit[orig_label_id]
                if orig_label_name.startswith("B-"):
                    label_ids.append(label2id_grit["I-" + orig_label_name[2:]])
                else:
                    label_ids.append(orig_label_id)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


tokenized_cahya = dataset_grit.map(
    tokenize_and_align_labels_cahya,
    batched=True,
    desc="Tokenisasi XLM-RoBERTa",
)
print(tokenized_cahya)

# ── Verifikasi: pastikan ada token entitas ────────────────────────────────
from collections import Counter
all_lbl = [l for row in tokenized_cahya["train"]["labels"] for l in row if l != -100]
ctr = Counter(all_lbl)
o_cnt = ctr.get(0, 0)
total = sum(ctr.values())
print(f"\nToken entitas: {total - o_cnt:,} / {total:,} ({(total-o_cnt)/total*100:.1f}%)")
assert total - o_cnt > 0, "❌ BUG: tidak ada token entitas!"
print("✅ Tokenisasi OK.")

Tokenisasi XLM-RoBERTa:   0%|          | 0/1672 [00:00<?, ? examples/s]

Tokenisasi XLM-RoBERTa:   0%|          | 0/209 [00:00<?, ? examples/s]

Tokenisasi XLM-RoBERTa:   0%|          | 0/209 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1672
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 209
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 209
    })
})

Token entitas: 16,580 / 76,591 (21.6%)
✅ Tokenisasi OK.


## Langkah 3.6 — Fungsi Evaluasi (seqeval)

Identik dengan Bagian 2 — seqeval strict IOB2. Menggunakan `label_list_grit` yang sama untuk konsistensi evaluasi.


In [64]:
seqeval_cahya = evaluate.load("seqeval")

def compute_metrics_cahya(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_grit[pred] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list_grit[lbl] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval_cahya.compute(
        predictions=true_predictions,
        references=true_labels,
        mode="strict",
        scheme="IOB2",
    )
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

## Langkah 3.7 — Model, Training Arguments, dan Pelatihan

### Penyesuaian Hyperparameter untuk Model Besar

Karena XLM-RoBERTa-Large **~4.5x lebih besar** dari IndoBERT, beberapa hyperparameter disesuaikan:

| Parameter | IndoBERT | XLM-RoBERTa-Large | Alasan |
|-----------|----------|-------------------|--------|
| `learning_rate` | 2e-5 | **1e-5** | Model besar lebih sensitif terhadap lr tinggi |
| `per_device_train_batch_size` | 16 | **8** | VRAM lebih banyak dipakai per sampel |
| `gradient_accumulation_steps` | — | **2** | Efektif batch size = 8 × 2 = 16 |
| `warmup_ratio` | — | **0.1** | Warmup penting untuk stabilitas model besar |
| `ignore_mismatched_sizes` | — | **True** | Classifier head baru berbeda ukuran dari pretrained |


In [66]:
# ── Model ─────────────────────────────────────────────────────────────────
model_cahya = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT_CAHYA,
    num_labels=len(label_list_grit),
    id2label=id2label_grit,
    label2id=label2id_grit,
    ignore_mismatched_sizes=True  # Tambahkan ini untuk mengabaikan perbedaan ukuran classifier
)

data_collator_cahya = DataCollatorForTokenClassification(tokenizer_cahya)

# ── Training arguments ────────────────────────────────────────────────────
# lr lebih kecil (1e-5) karena model lebih besar → lebih sensitif terhadap lr tinggi
training_args_cahya = TrainingArguments(
    output_dir="./xlmr-cahya-nergrit-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,        # lebih kecil dari IndoBERT karena model lebih besar
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,        # efektif batch size = 8 × 2 = 16
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,                     # warmup 10% steps — penting untuk model besar
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# ── Trainer ───────────────────────────────────────────────────────────────
trainer_cahya = Trainer(
    model=model_cahya,
    args=training_args_cahya,
    train_dataset=tokenized_cahya["train"],
    eval_dataset=tokenized_cahya["validation"],
    data_collator=data_collator_cahya,
    compute_metrics=compute_metrics_cahya,
)

trainer_cahya.train()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: cahya/xlm-roberta-large-indonesian-NER
Key                             | Status     |                                                                                        
--------------------------------+------------+----------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                        
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                        
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                        
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([39]) vs model:torch.Size([7])            
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([39, 102

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.378594,0.163637,0.833333,0.808157,0.820552,0.951763
2,0.284487,0.150306,0.834848,0.832326,0.833585,0.954929
3,0.218366,0.145410,0.841555,0.850453,0.845980,0.958096
4,0.169731,0.150032,0.833333,0.853474,0.843284,0.958201
5,0.152888,0.155173,0.831858,0.851964,0.841791,0.958201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=525, training_loss=0.3640380645933605, metrics={'train_runtime': 904.8517, 'train_samples_per_second': 9.239, 'train_steps_per_second': 0.58, 'total_flos': 1384711088274096.0, 'train_loss': 0.3640380645933605, 'epoch': 5.0})

## Langkah 3.8 — Simpan Model XLM-RoBERTa-Large

Model terbaik disimpan ke `./xlmr-cahya-nergrit-best/`.


In [ ]:
trainer_cahya.save_model("./xlmr-cahya-nergrit-best")
tokenizer_cahya.save_pretrained("./xlmr-cahya-nergrit-best")
print("Model XLM-RoBERTa (cahya) + NER-Grit tersimpan.")

## Langkah 3.9 — Inferensi dengan Model XLM-RoBERTa-Large

Model diuji dengan kalimat yang **sama persis** dengan Bagian 1 dan 2 untuk perbandingan yang adil.

Output menunjukkan entitas yang terdeteksi beserta label dan confidence score, memungkinkan analisis komparatif tiga model.


In [67]:
from transformers import pipeline

ner_pipeline_cahya = pipeline(
    "token-classification",
    model=model_cahya,
    tokenizer=tokenizer_cahya,
    aggregation_strategy="simple",
)

sample_sentences = [
    "Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.",
    "Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.",
    "Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung Kementerian Keuangan, Jakarta Pusat.",
    "Aturan kenaikan pajak hiburan tercantum dalam Undang-Undang Nomor 1 Tahun 2022 tentang Hubungan Keuangan antara Pemerintah Pusat dan Pemerintahan Daerah.",
    "Direktorat Jenderal Perimbangan Keuangan (DJPK) Kementerian Keuangan menyebut pengenaan besaran pajak tersebut karena penikmat hiburan karaoke hingga spa berasal dari masyarakat kalangan tertentu.",
    "Pembahasaan UU HKPD terkait pengenaan besaran pajak itu telah disetujui dan disepakati oleh Dewan Perwakilan Rakyat (DPR) RI.",
]

print("=" * 70)
print(f"{'Model: XLM-RoBERTa-Large (cahya) + NER-Grit':^70}")
print("=" * 70)

for i, sent in enumerate(sample_sentences, 1):
    results = ner_pipeline_cahya(sent)
    print(f"\n[Kalimat {i}] {sent}")
    if results:
        for ent in results:
            print(f"  → {ent['entity_group']:<6}  {ent['word']:<35}  score: {ent['score']:.4f}")
    else:
        print("  → (tidak ada entitas terdeteksi)")

             Model: XLM-RoBERTa-Large (cahya) + NER-Grit              

[Kalimat 1] Pemilik Beach Club Atlas, Hotman Paris Hutapea memberikan pesan kepada Menteri Keuangan, Sri Mulyani Indrawati.
  → LOC     Beach Club Atlas                     score: 0.9202
  → PER     Hotman                               score: 0.5317
  → PER     Paris Hutapea                        score: 0.9326
  → PER     Sri Mulyani Indrawati                score: 0.9915

[Kalimat 2] Ujar Hotman kepada media, di Kantor Kementerian Koordinator Bidang Kemaritiman dan Investasi, Jakarta.
  → PER     Hotman                               score: 0.9870
  → LOC     Kantor                               score: 0.4932
  → LOC     Koordinator Bidang Kemaritiman dan Investasi  score: 0.5664
  → LOC     Jakarta                              score: 0.9775

[Kalimat 3] Direktur Pajak Daerah dan Retribusi Daerah DJPK, Lydia Kurniawati Christyana dalam Media Briefing di Gedung Kementerian Keuangan, Jakarta Pusat.
  → ORG     DJPK 

---

# Ringkasan Eksperimen

## Perbandingan Tiga Eksperimen

| Eksperimen | Model | Dataset | Keunggulan |
|------------|-------|---------|------------|
| 1 | IndoBERT-base | WikiANN (ID) | Baseline, dataset mudah diakses via HF Hub |
| 2 | IndoBERT-base | NER-Grit | Domain berita Indonesia, label lebih spesifik |
| 3 | XLM-RoBERTa-Large | NER-Grit | Model paling kuat, multilingual backbone |

## Alur Pipeline NER (Umum)

```
Teks Input
    ↓
Tokenisasi (WordPiece / SentencePiece)
    ↓
Alignment Label (IOB2)
    ↓
Fine-tuning Model (Transformer)
    ↓
Prediksi per Token
    ↓
Agregasi Entitas (aggregation_strategy)
    ↓
Output: [Entitas, Tipe, Score]
```

## Kesimpulan
- **Fine-tuning** memungkinkan model pretrained diadaptasi untuk tugas NER spesifik dengan data dan komputasi minimal.
- **Dataset** berpengaruh besar — NER-Grit lebih relevan untuk teks berita Indonesia.
- **Model besar** (XLM-RoBERTa-Large) umumnya menghasilkan F1 lebih tinggi dengan biaya komputasi yang lebih besar.
- **IOB2 alignment** yang konsisten adalah kunci akurasi pelatihan NER.
